# Exploración — Mapa Coroplético

Notebook de exploración para el mapa de la Región de Aysén.

Objetivo: cargar el shapefile comunal, unirlo con datos de población (INE) y conexiones (SUBTEL), y explorar la visualización con Plotly antes de implementarla en `app/charts/mapa.py`.

In [ ]:
import geopandas as gpd
import pandas as pd
import plotly.express as px
import json
import sys
sys.path.insert(0, '../app')
import loaders

# Rutas
SHP_PATH = '../data/raw/SHP_R11/Comunal.shp'
INE_PATH = '../data/raw/ine_proyecciones_comunas.xlsx'

## 1. Cargar shapefile y explorar estructura

In [ ]:
# Cargar shapefile — ya viene filtrado para la Región de Aysén (R11)
gdf = gpd.read_file(SHP_PATH)

print('Shape:', gdf.shape)
print('CRS:', gdf.crs)  # sistema de coordenadas
gdf[['CUT', 'N_COMUNA', 'N_PROVINCI']]

In [ ]:
# Reproyectar a WGS84 (EPSG:4326) que es lo que requiere Plotly
gdf = gdf.to_crs(epsg=4326)
gdf['CUT'] = gdf['CUT'].astype(int)
print('CRS después de reproyectar:', gdf.crs)

## 2. Diccionario CUT → nombre correcto de cada comuna

El shapefile tiene problemas de encoding en los caracteres especiales (`AYSA0N` en vez de `Aysén`, `RADO IBADAÑEZ` en vez de `Río Ibáñez`). En vez de intentar corregir el encoding, se usa un diccionario con el CUT comunal como clave — un identificador numérico único que no tiene problemas de encoding — para asignar el nombre correcto a cada comuna.

Esto también hace que el join entre el shapefile y los datos de SUBTEL e INE sea más robusto, ya que se evitan comparaciones de strings con tildes y mayúsculas.

In [ ]:
# Diccionario CUT → nombre correcto de cada comuna
NOMBRES_COMUNAS = {
    11101: 'Coyhaique',
    11102: 'Lago Verde',
    11201: 'Aysén',
    11202: 'Cisnes',
    11203: 'Guaitecas',
    11301: 'Cochrane',
    11302: "O'Higgins",
    11303: 'Tortel',
    11401: 'Chile Chico',
    11402: 'Río Ibáñez',
}

# Asignar nombre correcto usando el diccionario
gdf['nombre_comuna'] = gdf['CUT'].map(NOMBRES_COMUNAS)

# Simplificar geometría para reducir tiempo de renderizado
gdf['geometry'] = gdf['geometry'].simplify(tolerance=0.001, preserve_topology=True)

# Convertir a GeoJSON
geojson = json.loads(gdf.to_json())

print('Nombres corregidos:')
gdf[['CUT', 'N_COMUNA', 'nombre_comuna']]

## 3. Preparar datos de población (INE)

Usamos el año 2024 como referencia para el mapa de población.

In [ ]:
df_ine = pd.read_excel(INE_PATH, sheet_name='Est. y Proy. de Pob. Comunal')

# Filtrar Aysén y agrupar por comuna y año (sumar sexo y edad)
df_pob = (
    df_ine.query('Region == 11')
    .groupby(['Comuna', 'Nombre Comuna'], as_index=False)['Poblacion 2024']
    .sum()
    .rename(columns={'Poblacion 2024': 'poblacion', 'Comuna': 'CUT', 'Nombre Comuna': 'comuna'})
)

# El CUT debe ser entero para hacer el join con el shapefile
df_pob['CUT'] = df_pob['CUT'].astype(int)

print('Shape:', df_pob.shape)
df_pob

## 4. Preparar datos de conectividad (SUBTEL)

Usamos el último período disponible (diciembre 2025).

In [ ]:
CUT_POR_NOMBRE = {v: k for k, v in NOMBRES_COMUNAS.items()}

# Cargar datos de conectividad y filtrar diciembre 2025
df_con = loaders.load_conectividad_aysen()
df_con_2025 = df_con.query('anio == 2025 and mes == "Dic"')[['comuna', 'conexiones']].copy()

# Agregar CUT a los datos de conectividad
df_con_2025['CUT'] = df_con_2025['comuna'].map(CUT_POR_NOMBRE)

df_con_2025

## 5. Join shapefile + datos

Usamos el CUT como clave de unión (más robusto que el nombre, evita problemas de tildes y mayúsculas).

In [ ]:
# Join por CUT: más robusto que por nombre (evita problemas de tildes y encoding)
gdf_merged = gdf.merge(df_pob, on='CUT', how='left')
gdf_merged = gdf_merged.merge(df_con_2025[['CUT', 'conexiones']], on='CUT', how='left')

# Conexiones per cápita (por cada 100 habitantes)
gdf_merged['conexiones_per_capita'] = (
    (gdf_merged['conexiones'] / gdf_merged['poblacion'] * 100).round(2)
)

# Verificar que no haya NaN inesperados
print('NaN por columna:')
print(gdf_merged[['nombre_comuna', 'poblacion', 'conexiones', 'conexiones_per_capita']].isnull().sum())
print()
gdf_merged[['nombre_comuna', 'CUT', 'poblacion', 'conexiones', 'conexiones_per_capita']]

## 6. Mapa coroplético con Plotly

In [ ]:
# Centroide de la región para centrar el mapa
CENTRO_LAT = gdf_merged.geometry.centroid.y.mean()
CENTRO_LON = gdf_merged.geometry.centroid.x.mean()
print(f'Centro: lat={CENTRO_LAT:.2f}, lon={CENTRO_LON:.2f}')

In [ ]:
# Mapa 1: Población 2024
fig = px.choropleth_mapbox(
    gdf_merged,
    geojson=geojson,
    locations='CUT',
    featureidkey='properties.CUT',
    color='poblacion',
    hover_name='nombre_comuna',
    hover_data={'poblacion': ':,.0f', 'conexiones': ':,.0f',
                'conexiones_per_capita': ':.1f', 'CUT': False},
    color_continuous_scale='Blues',
    mapbox_style='carto-positron',
    zoom=5.5,
    center={'lat': CENTRO_LAT, 'lon': CENTRO_LON},
    opacity=0.75,
    title='Población por comuna — Región de Aysén (2024)'
)
fig.update_layout(margin=dict(l=0, r=0, t=40, b=0))
fig.show()

In [ ]:
# Mapa 2: Conexiones de internet (de diciembre 2025)
fig = px.choropleth_mapbox(
    gdf_merged,
    geojson=geojson,
    locations='CUT',
    featureidkey='properties.CUT',
    color='conexiones',
    hover_name='nombre_comuna',
    hover_data={'poblacion': ':,.0f', 'conexiones': ':,.0f',
                'conexiones_per_capita': ':.1f', 'CUT': False},
    color_continuous_scale='Oranges',
    mapbox_style='carto-positron',
    zoom=5.5,
    center={'lat': CENTRO_LAT, 'lon': CENTRO_LON},
    opacity=0.75,
    title='Conexiones de internet residencial — Región de Aysén (Diciembre 2025)'
)
fig.update_layout(margin=dict(l=0, r=0, t=40, b=0))
fig.show()

In [ ]:
# Mapa 3: Conexiones per Cápita (por cada 100 habitantes)
fig = px.choropleth_mapbox(
    gdf_merged,
    geojson=geojson,
    locations='CUT',
    featureidkey='properties.CUT',
    color='conexiones_per_capita',
    hover_name='nombre_comuna',
    hover_data={'poblacion': ':,.0f', 'conexiones': ':,.0f',
                'conexiones_per_capita': ':.1f', 'CUT': False},
    color_continuous_scale='Purples',
    mapbox_style='carto-positron',
    zoom=5.5,
    center={'lat': CENTRO_LAT, 'lon': CENTRO_LON},
    opacity=0.75,
    title='Conexiones residenciales por cada 100 hab. — Región de Aysén (2025)'
)
fig.update_layout(margin=dict(l=0, r=0, t=40, b=0))
fig.show()